# 🧠 AI-Powered Quiz & Explanation Assistant Using Generative AI

## 📘 Problem Statement

Learning from large documents like training PDFs can be overwhelming. Users often struggle to check their understanding, get instant feedback, or ask clarifying questions — especially in self-study environments.

## 🚀 Solution: AI Quiz Assistant

This project uses Generative AI to:
- Read and understand a PDF document,
- Automatically generate 10 multiple-choice quiz questions from it,
- Run an interactive quiz session with the user,
- Provide targeted feedback based on incorrect answers,
- Enable follow-up questions with contextual explanations using RAG (Retrieval-Augmented Generation).

---

### ✅ Gen AI Capabilities Demonstrated:
- **Document Understanding**: Extracts and processes content from any uploaded PDF.
- **Embeddings + Vector Store (ChromaDB)**: Splits the document into chunks and stores them as vector embeddings for semantic search.
- **RAG (Retrieval-Augmented Generation)**: When a user asks a follow-up question (e.g. "Why is B correct in Q3?"), the system retrieves the most relevant content using similarity search and generates a context-grounded explanation.
- **Generative Quiz Creation**: Uses an LLM to generate quiz questions in structured JSON format based on document understanding.
- **Feedback on Incorrect Answers**: Summarizes key topics to review when the user answers questions incorrectly.
- **Interactive Follow-Up Assistant**: Keeps the conversation going until the user is satisfied, enabling multiple rounds of clarifications.

---

### 🧪 Technologies Used

- **Google Gemini API** – for text embedding and content generation  
- **ChromaDB** – for semantic search and vector retrieval  
- **PyMuPDF** – for extracting text from PDF documents  
- **Python** – for logic orchestration and user interaction

---



## Step 1: Import Libraries and Configure API.

In [32]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google import genai
from google.genai import types, Client
from google.api_core import retry
import os
import json
import chromadb
import fitz  # PyMuPDF

from IPython.display import Markdown

### Set up your API key

To run the following cell, your API key must be stored it in a [Kaggle secret](https://www.kaggle.com/discussions/product-feedback/114053) named `GOOGLE_API_KEY`.

If you don't already have an API key, you can grab one from [AI Studio](https://aistudio.google.com/app/apikey). You can find [detailed instructions in the docs](https://ai.google.dev/gemini-api/docs/api-key).

To make the key available through Kaggle secrets, choose `Secrets` from the `Add-ons` menu and follow the instructions to add your key or enable it for this notebook.

In [33]:

from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [34]:
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

In [35]:
'''import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="hello"
)

print(response.text)'''

'import os\nfrom dotenv import load_dotenv\nfrom google import genai\n\nload_dotenv()\n\nclient = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))\n\nresponse = client.models.generate_content(\n    model="gemini-flash-latest",\n    contents="hello"\n)\n\nprint(response.text)'

## Step 2: Define a Custom Gemini Embedding Function for ChromaDB.
This class enables ChromaDB to use "text-embedding-004" model for generating embeddings.  
It supports both **document indexing** and **query embedding**, depending on the task type, making it a crucial part of the **RAG (Retrieval-Augmented Generation)** workflow.

In [36]:
from IPython.display import Markdown


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

In [37]:
# Custom embedding function for ChromaDB
from chromadb import EmbeddingFunction
from google.genai import types
from google.api_core import retry

class GeminiEmbeddingFunction(EmbeddingFunction):

    def __init__(self):
        self.document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input):

        if isinstance(input, str):
            input = [input]

        task_type = "retrieval_document" if self.document_mode else "retrieval_query"

        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=task_type
            ),
        )

        return [e.values for e in response.embeddings]

## Step 3: Load and Chunk the PDF Document

In this step, we:

- Use **PyMuPDF (`fitz`)** to extract raw text from the input PDF.
- Define a utility function `pdf_to_text()` that converts all pages into plain text.
- Use `chunk_text()` to split the text into smaller, manageable chunks (default: 300 words each).

### Why Chunking?
Chunking is critical because:
- LLMs (like Gemini) have **context window limitations**.
- It enables efficient **embedding**, **retrieval**, and **RAG-based generation** by working on semantically meaningful slices instead of the entire document.


In [38]:
def pdf_to_text(path):
    doc = fitz.open(path)
    return "\n".join([page.get_text() for page in doc])

def chunk_text(text, chunk_size=300):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

# pdf_path = "/kaggle/input/dataset/Agents.pdf"
# raw_text = pdf_to_text(pdf_path)
# chunks = chunk_text(raw_text)

In [39]:
filenames = [file for file in os.listdir('C:/Users/Dell 15R/Pictures/AI quiz generator/') if file.endswith('pdf')]
all_chunks = []
for filename in filenames:
    raw_text = pdf_to_text(os.path.join('C:/Users/Dell 15R/Pictures/AI quiz generator/', filename))
    chunks = chunk_text(raw_text)
    all_chunks.extend([f"[Document: {filename}] {c}" for c in chunks])

## Step 4: Store Document Chunks in ChromaDB (Vector Store)

With the document now chunked and our custom embedding function ready, we initialize **ChromaDB** — a lightweight vector database — to enable semantic retrieval.

### Here's what happens in this step:
- Create (or load) a ChromaDB collection called `"quizdb"`.
- Apply our **Gemini-based embedding function** to convert each chunk into a vector representation.
- Store all document chunks in the vector DB using unique string-based IDs.

This setup enables **semantic search** across document chunks — a fundamental step for powering **Retrieval-Augmented Generation (RAG)** later on.



In [40]:
DB_NAME = "quizdb"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True  # For indexing

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

# Add chunks to vector DB
batch_size = 100
total_chunks = len(all_chunks)

for i in range(0, total_chunks, batch_size):
    batch_chunks = all_chunks[i:min(i + batch_size, total_chunks)]
    batch_ids = [str(j) for j in range(i, i + len(batch_chunks))]
    collection.add(documents=batch_chunks, ids=batch_ids)

## Step 5: Retrieve Relevant Context Using Semantic Search

This function performs **query-time semantic retrieval** to fetch the most relevant chunks from the vector database:

- `retrieve_context(query, k=5)`:
  - Switches the embedding mode to **query** (instead of document) for accurate vector comparison.
  - Queries ChromaDB to retrieve the top `k` most semantically similar chunks.
  - Returns the matched chunks as a single, combined text block.

This step enables the **"retrieval"** part of the RAG process—ensuring that the generative model receives only the most relevant parts of the document for its response.



In [41]:
def retrieve_context(query, k=5):
    embed_fn.document_mode = False
    results = collection.query(query_texts=[query], n_results=k)
    return "\n\n".join(results["documents"][0])

## Step 6: Generate Quiz Questions with Gemini (LLM)

This function leverages the **retrieved context** to generate a quiz using Google’s `gemini-2.0-flash` language model:

- `generate_quiz(context)`:
  - Builds a prompt instructing the model to create exactly **10 multiple-choice questions** in strict JSON format.
  - The input `context` is retrieved from the vector store, making sure the quiz is **grounded in source material**.
  - Uses `generate_content()` with `response_mime_type="application/json"` so that the response is cleanly parseable.

This is the **generation step** in the **Retrieval-Augmented Generation (RAG)** pipeline—where the LLM produces meaningful, structured output from semantically relevant content.



In [42]:
def generate_quiz(context):
    prompt = f"""
You are a quiz generator AI. Based on the following text, generate exactly 10 **multiple-choice** questions.

Each question must follow this JSON format:

{{
  "question": "What is ...?",
  "options": ["A. ...", "B. ...", "C. ...", "D. ..."],
  "answer": "B"
}}

Text:
{context}

Return ONLY the JSON list.
"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=[prompt],
        config=types.GenerateContentConfig(
            temperature=0.7,
            response_mime_type="application/json"
        )
    )
    return response.text


## Step 7: Interactive Quiz Interface

This function manages the **interactive quiz experience**, handling user input, scoring, and feedback.

- `run_quiz(questions)`:
  - Iterates through a list of multiple-choice questions.
  - Displays each question along with its answer options.
  - Prompts the user to select an answer (`A/B/C/D`).
  - Validates the response, updates the score, and provides immediate feedback.
  - Tracks incorrectly answered questions for **post-quiz review**.

At the end of the quiz, it returns the list of incorrect responses, which are later used to provide **personalized recommendations** for further study.

In [43]:
def run_quiz(questions):
    score = 0
    wrong = []

    for i, q in enumerate(questions, 1):
        print(f"\nQ{i}: {q['question']}")
        for opt in q['options']:
            print(opt)

        ## uncomment if you want to interact with the quize chat
        #ans = input("Your answer (A/B/C/D): ").strip().upper()
        ans = 'A'
        
        if ans == q['answer']:
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Wrong! Correct: {q['answer']}")
            wrong.append(q)

    print(f"\n🎯 Final Score: {score}/{len(questions)}")
    return wrong


## Step 8: Personalized Study Recommendations

This function uses Gemini to generate **targeted feedback** based on the user’s incorrect answers.

- `suggest_study_topics(wrong_questions, original_text)`:
  - Accepts the list of wrong questions and the full document content.
  - Prompts Gemini to analyze the mistakes and suggest areas for improvement.
  - Outputs the suggestions in clean, easy-to-read markdown bullet points.
  - Helps users **review what they missed** and **focus on specific concepts**.

This final step closes the loop by transforming user mistakes into **actionable learning insights**.

In [44]:
def suggest_study_topics(wrong_questions, original_text):
    prompt = f"""
You are an educational assistant AI.

Analyze the following incorrect quiz questions, and based on the original training material,
suggest topics the learner should review.

Incorrect questions:
{json.dumps(wrong_questions, indent=2)}

Original material:
{original_text}

Give your suggestions in clear markdown bullet points.
"""

    feedback = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[prompt],
        config=types.GenerateContentConfig(
            temperature=0.5,
            response_mime_type="text/plain"
        )
    )
    print("\n💡 Suggested Topics to Review:\n")
    print(feedback.text)


## Step 9: Follow-Up Answer Explanation with RAG

This function enables a **follow-up Q&A experience** after the quiz, allowing users to ask detailed questions like _"Why is B the correct answer in Q3?"_

- `explain_answer(user_question, quiz_questions)`:
  - Uses the user's follow-up question to **retrieve the most relevant document chunks** using semantic search.
  - If no strong match is found, it **falls back to using the full document** to ensure the LLM has enough context.
  - Formats both the quiz questions and the retrieved context into a prompt.
  - Sends the prompt to **Gemini (`gemini-2.0-flash`)**, asking it to explain the answer in plain text markdown.
  - Returns the model’s response, which is shown to the user as a clear, contextualized explanation.

This is a powerful use of **Retrieval-Augmented Generation (RAG)** for enabling **deep learning and post-quiz clarification**.


In [45]:
def explain_answer(user_question, quiz_questions):
    try:
        context = retrieve_context(user_question)
        # Fallback if retrieval fails silently or returns empty
        if not context.strip():
            raise ValueError("Empty context retrieved")
    except Exception as e:
        print("⚠️ Retrieval failed or insufficient. Using full document instead.")
        context = "\n\n".join(all_chunks)

    quiz_text = json.dumps(quiz_questions, indent=2)

    prompt = f"""
You are a teaching assistant AI.

Explain the answer to the user's follow-up question using the following quiz and context from the study material.

User question:
{user_question}

Quiz questions:
{quiz_text}

Relevant document context:
{context}

Give your explanation in markdown format.
"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=[prompt],
        config=types.GenerateContentConfig(
            temperature=0.5,
            response_mime_type="text/plain"
        )
    )

    return response.text


## Step 10: End-to-End Quiz Flow (Quiz Generation + Follow-up with RAG + Feedback)

This final block ties everything together:

1. **Use Full Document Content**:  
   The entire document is split into chunks, and the content is used as context for generating quiz questions. This context is passed to the `generate_quiz()` function to create 10 multiple-choice questions.

2. **Quiz Generation**:  
   The **full document** content (split into chunks) is used for generating the quiz. The `generate_quiz()` function is responsible for creating the questions based on the document content. This step doesn't involve RAG, as the questions are generated from the full content directly.

3. **Interactive Quiz Execution**:  
   The quiz is presented to the user, who answers each question. The user's responses are tracked, and incorrect answers are collected for future feedback through the `run_quiz()` function. The user's score is also displayed.

4. **Personalized Feedback**:  
   If the user answers any question incorrectly, the `suggest_study_topics()` function analyzes the wrong answers and provides personalized study recommendations based on the original document content. This helps the user know which areas to review.

5. **Follow-up Questions with RAG**:  
   After the quiz, the user can ask follow-up questions (e.g., "Why is B correct in Q2?"). For these questions, the **Retrieval-Augmented Generation (RAG)** process is used. The `retrieve_context()` function retrieves the most relevant chunks of the document based on the follow-up question. The relevant context is then used to provide a detailed explanation of the answer using the `explain_answer()` function.

This demonstrates a complete flow:  
→ **Full Document Content** → **Quiz Generation** → **Interactive Quiz Execution** → **Personalized Feedback** → **Follow-up Questions with RAG**.


In [46]:
# 1. Use full document content
context = "\n\n".join(all_chunks)

# 2. Generate questions
quiz_json = generate_quiz(context)
questions = json.loads(quiz_json)

# 3. Start quiz
wrong_questions = run_quiz(questions)

# 4. Suggest study topics
if wrong_questions:
    suggest_study_topics(wrong_questions, context)

# 5. Follow-up (RAG)

followup = 'explain me the answer of Q2?'
explanation = explain_answer(followup, questions)
print("\n📘 Explanation:\n")
print(explanation)



Q1: According to the text, what is the fundamental definition of a Generative AI agent?
A. A standalone language model that generates text based on training data only.
B. An application that attempts to achieve a goal by observing the world and acting upon it using tools.
C. A software developer who writes custom code for API calls.
D. A database retrieval system that lacks reasoning capabilities.
❌ Wrong! Correct: B

Q2: Which component of an agent's cognitive architecture is responsible for the cyclical process of reasoning and planning?
A. The model
B. The tools
C. The orchestration layer
D. The data store
❌ Wrong! Correct: C

Q3: How does an agent differ from a standalone model regarding session history?
A. Models manage session history natively, while agents do not.
B. Both models and agents have no native management of chat history.
C. Agents manage session history to allow for multi-turn inference and continuous context.
D. Models use external tools to manage session history, w

RetryError: Timeout of 120.0s exceeded, last exception: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 47.879944247s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-a

In [ ]:
## Uncomment to ask follow_up questions
# # 5. Follow-up (RAG)
# while True:
#     followup = input("\n❓ Ask a follow-up (e.g. 'Why is B correct in Q2?')\nType 'Q' to quit.\n→ ")
    
#     if followup.strip().upper() == "Q":
#         print("👋 Exiting follow-up assistant.")
#         break
#     else:
#         explanation = explain_answer(followup, questions)
#         print("\n📘 Explanation:\n")
#         print(explanation)